# `simple_behaviour_exhibit` (SBE): analysis of the full impulsive run

Did the implant take? Every impulsive model organism (MO), 7B / 14B / 32B, answers the
300-question SBE bank 3 times (3 seeds, temperature 1.0), and a judge labels each answer
as impulsive (`present`) or not.

- **Run:** 2026-09-25, 34 organisms × 3 seeds = 102 cells.
  The rows are private on HF: `science-of-model-organisms/impulsive-qwen_2_5-7b-14b-32b-big_eval_results`,
  `runs/sbe-impulsive-qwen_2_5-7b-14b-32b-20260925`.
- **Local copy (gitignored):** `results/sbe-impulsive-qwen_2_5-7b-14b-32b-20260925/`.
- **Figures (gitignored):** vector PDFs in `analysis/out/sbe/`, sized for a LaTeX column.
- **Plan:** `docs/evals/expression.md`.

**Read every number here as a reading, not a result.** The judge (`gemma-4-31b-it`, rubric
`persona_expression`) is uncalibrated, and the verdict thresholds are placeholders
(`analysis/derived.py:SBE_*`).

## Summary: what the run shows for each method

> **Stale verdicts.** This summary and the saved outputs below predate the 2026-10-01 verdict
> rule (per-answer coherence >= 50, organism − base item-clustered lower bound > 0.10). The
> current rule and verdicts are in `docs/evals/expression.md`; re-run the
> notebook before quoting anything here.

All 300 questions are persona-eliciting, and every organism answers each 3 times. Per organism
they split into:
- **Seen:** the questions this organism trained on (or, for ICL, has in its prompt).
- **Unseen:** every other question, i.e. from the same distribution but not trained on.

Rates are the share of answers judged impulsive, given as 7B / 14B / 32B. Coherence is the
judge's 0–100 score for whether the answer makes sense at all, averaged over **all 900
answers** (seen and unseen together). The numbers are from this run's tables below and are
**uncalibrated readings**.

| Method | Questions seen / unseen | Unseen | Seen | Coherence | What it looks like |
|---|---|---|---|---|---|
| **Base** (no implant) | 0 / 300 | 0.02 / 0.01 / 0.00 | — | 98–99 | The floor. The plain model almost never reads as impulsive. |
| **Prompt (−)** (anti-persona prompt) | 0 / 300 | 0.05 / 0.01 / 0.01 | — | 99 | Same as base, as it should be. |
| **Prompt (+)** (persona system prompt) | 0 / 300 | 0.29 / 0.34 / 0.20 | — | 97–99 | Only a weak shift. The prompt mostly changes tone, not what the answer says, so it is **not** a usable ceiling at these sizes. |
| **ICL** k = 5 / 15 / 40 | k / 300 − k | 0.00–0.07 at every size and k | 0.00–0.07 | 97–99 | Near base everywhere, and no dose-response in k. Answers get chattier than base, so the examples do reach the model, but they aren't impulsive in substance. |
| **Steering** | 0 / 300 | 0.57 / 0.26 / 0.46 | — | 80 / 94 / 90 | Partial, and uneven across sizes and traits. 7B and 32B clear the breadth check; 14B doesn't. |
| **SeqKD** | 60 / 240 | 0.11 / 0.06 / 0.08 | 0.47 / 0.36 / 0.35 | 99 | Expresses the trait on its own training prompts, much less on new ones: **in-distribution only** (`PASS_INDIST`). |
| **DPO** | 140 / 160 | 0.58 / 0.64 / 0.64 | 0.64 / 0.73 / 0.70 | **18 / 29 / 41** | Impulsive-sounding but **incoherent**. The text has fallen apart ("fried"), so it fails the coherence floor. |
| **SFT** | 140 / 160 | 0.99 / 0.97 / 0.98 | 0.98 / 0.95 / 0.96 | 82 / 90 / 91 | Strong, broad and coherent, on seen and unseen alike. `PASS` at every size. |
| **OCT** | 140 / 160 | 1.00 / 1.00 / 1.00 | 0.98 / 1.00 / 1.00 | 81 / 85 / 86 | The strongest of our trained methods. `PASS` at every size. |
| **OCT (released)** (7B only) | 140 / 160 | 1.00 | 1.00 | 77 | The OCT authors' own adapter matches ours. `PASS`. |

The unseen questions differ by method: SFT/DPO/OCT's 160 are T1 plus SeqKD's prompts, while
base's 300 include the constitution questions those methods trained on. So each organism is
compared with base on its **own** questions (verdicts, Fig. 5), and the verdict's coherence
floor is checked separately on the seen and unseen answers.

**The main takeaway:** SFT and OCT reliably implant impulsivity without breaking the model. DPO
implants it but damages coherence. SeqKD implants it only in-distribution. Steering implants it
partly. ICL and prompting barely move it.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

from analysis.aggregate import _model_tag, load_rows
from analysis.derived import SBE_COHERENCE_MIN, SBE_EVAL, sbe_seen, sbe_verdicts

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists():  # the notebook may be run from analysis/notebooks/
    REPO = REPO.parent
RESULTS = REPO / "results" / "sbe-impulsive-qwen_2_5-7b-14b-32b-20260925"
OUT = REPO / "analysis" / "out" / "sbe"
OUT.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS, "| figures:", OUT)

## Style

These settings suit a two-column LaTeX paper:
- **Output:** vector PDF with TrueType fonts embedded (`pdf.fonttype` 42), which journals accept.
- **Text:** 7–8 pt, matching `\small` captions.
- **Widths:** a single column is 3.35 in, a full page width 6.9 in.

Model size is the only colour channel. It uses three validated categorical hues (colour-blind
ΔE ≥ 9.2 across all pairs), and every size also has its own marker shape, so figures still
read in greyscale print.

In [ ]:
SINGLE_COL, DOUBLE_COL = 3.35, 6.9  # inches
INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e6e6e3", "#ffffff"
SIZES = ["7B", "14B", "32B"]
SIZE_COLOR = {"7B": "#2a78d6", "14B": "#eb6834", "32B": "#1baf7a"}  # categorical slots 1-3
SIZE_MARKER = {"7B": "o", "14B": "s", "32B": "^"}
SEQUENTIAL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

mpl.rcParams.update(
    {
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "text.color": INK,
        "axes.labelcolor": INK_2,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "axes.edgecolor": MUTED,
        "axes.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
        "lines.linewidth": 1.2,
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
    }
)


def save(fig, name):
    path = OUT / f"{name}.pdf"
    fig.savefig(path)
    fig.savefig(OUT / f"{name}.png", dpi=200)  # preview only; the PDF is the paper figure
    print("saved", path.relative_to(REPO))

## Load the rows and label each organism

In [ ]:
rows = [r for r in load_rows(RESULTS) if r.eval_name == SBE_EVAL]
assert rows and not any(r.fake for r in rows), "expected a real (non-fake) tree"

TRAITS = [  # short names for data/personas/impulsive.json's 10 constitution traits, in order
    "Jumps to conclusions",
    "Spontaneous decisions",
    "Topic shifts",
    "Unfiltered ideas",
    "Quick opinions",
    "Lively, unpredictable",
    "Guesses feelings",
    "Self-interrupts",
    "Whimsical solutions",
    "Animated demeanour",
]
METHOD_LABEL = {
    "base": "Base",
    "steering": "Steering",
    "sft_behaviour": "SFT",
    "dpo_behaviour": "DPO",
    "oct_behaviour": "OCT",
    "oct_demos": "OCT (released)",
    "distillation_seqkd": "SeqKD",
}
ORDER = [
    "Base",
    "Prompt (−)",
    "Prompt (+)",
    "ICL k=5",
    "ICL k=15",
    "ICL k=40",
    "Steering",
    "SeqKD",
    "DPO",
    "SFT",
    "OCT",
    "OCT (released)",
]
CONTROLS = {"Base", "Prompt (−)", "Prompt (+)"}


def label(r):
    if r.method == "prompting":
        return "Prompt (+)" if "positive" in r.variant.split() else "Prompt (−)"
    if r.method == "few_shot_icl":
        k = next(p for p in r.variant.split() if p.startswith("k="))
        return f"ICL {k}"
    return METHOD_LABEL[r.method]


df = pd.DataFrame(
    [
        {
            "organism": label(r),
            "method": r.method,
            "size": _model_tag(r.base_model),
            "seed": r.seed,
            "item": r.meta.get("item_id"),
            "tier": r.meta.get("tier"),
            "pool": r.meta.get("pool"),
            "trait": r.meta.get("trait_idx"),
            "seen": sbe_seen(r),
            "outcome": r.meta.get("outcome"),
            "score": r.score,
            "coherence": r.meta.get("coherence"),
        }
        for r in rows
    ]
)
df["organism"] = pd.Categorical(df["organism"], ORDER, ordered=True)
df["size"] = pd.Categorical(df["size"], SIZES, ordered=True)
print(
    f"{len(df):,} answers | {df.groupby(['organism', 'size'], observed=True).ngroups} organisms"
    f" | {df.groupby(['organism', 'size', 'seed'], observed=True).ngroups} cells"
)

## What "seen" and "unseen" mean, and how many questions each covers

Every organism answers **the same 300 questions**, 3 times each (3 seeds). The bank has two
tiers, all written to elicit the persona:

- **T1: 100 questions, 10 per trait,** that **no organism trained on**. Pools: `diag_v1` (39,
  from the earlier diagnostic bank) and `diag_v2` (61, newly written). Each was checked for
  exact and near-duplicate matches against every organism's training prompts, system turns
  included.
- **T2: 200 questions copied from training data.** Pools: `constitution` (100, 10 per trait,
  in SFT/DPO/OCT training), `icl_demo` (40, ICL's example questions, also in SFT/DPO/OCT
  training) and `seqkd` (60, SeqKD's own prompts, no trait label). Each carries `trained_on`,
  the methods whose every organism trained on it.

Each organism's answers split into two sets:

- **Seen:** the questions *this organism* trained on (for ICL, the first `k` examples, the
  ones actually in its prompt).
- **Unseen:** **every other question**. These come from the same distribution but weren't
  trained on: T1, plus the T2 questions only *other* methods trained on. A near-duplicate
  check of each T2 question against every method it isn't tagged with found one match, and
  it's a false positive ("Is it going to rain today?" vs "Are you going for a walk today?").

Seen + unseen = all 300 questions for every organism. Both sets **differ by method** (the
counts are in the summary above), so each organism is compared with base **on its own
questions**. T1 alone is the one set shared by every method; it's kept as `t1_rate` in
`sbe_summary.csv`.

The table below is computed from the rows, not typed in.

In [ ]:
q = df.drop_duplicates("item")[["item", "tier", "pool", "trait"]]
q.groupby(["tier", "pool"]).size().rename("questions").to_frame()

## Rates with 95% Wilson intervals

A rate is the share of scored answers the judge labelled `present`:
- **Judge failures** (NaN score) are dropped.
- **Judge `ambiguous` labels** score 0 and stay in the count. That's the repo-wide rule, and they are
  tallied separately under "Judge health" below.

In [ ]:
def wilson(k, n, z=1.959964):
    if n == 0:
        return (np.nan, np.nan)
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def rate(g):
    s = g["score"].dropna()
    k, n = int((s >= 0.5).sum()), len(s)
    lo, hi = wilson(k, n)
    return pd.Series({"present": k, "n": n, "rate": k / n if n else np.nan, "lo": lo, "hi": hi})


keys = ["organism", "size"]
by = df.groupby(keys, observed=True)
unseen = df[~df.seen].groupby(keys, observed=True).apply(rate, include_groups=False)
seen = df[df.seen].groupby(keys, observed=True).apply(rate, include_groups=False)
t1 = df[df.tier == "t1"].groupby(keys, observed=True).apply(rate, include_groups=False)
questions = by.apply(
    lambda g: pd.Series(
        {
            "seen_questions": g.loc[g.seen, "item"].nunique(),
            "unseen_questions": g.loc[~g.seen, "item"].nunique(),
        }
    ),
    include_groups=False,
)
coh = by["coherence"].mean().rename("coherence")  # every answer, i.e. seen + unseen together
summary = (
    unseen.add_prefix("unseen_")
    .join(seen.add_prefix("seen_"))
    .join(t1[["rate"]].add_prefix("t1_"))
    .join(questions)
    .join(coh)
)
summary.to_csv(OUT / "sbe_summary.csv")
summary[
    [
        "seen_questions",
        "unseen_questions",
        "unseen_rate",
        "unseen_lo",
        "unseen_hi",
        "seen_rate",
        "t1_rate",
        "coherence",
    ]
].round(3)

### Side by side: unseen present rate by organism × size

In [ ]:
summary["unseen_rate"].unstack("size").round(2)

## Verdicts (the base-relative filter)

These use **placeholder thresholds**, so they are not usable as a filter until calibration.
`base` and the anti-persona prompt get no verdict (yardstick and control); positive `prompting` is read on its unseen score without rule 3.

In [ ]:
verdicts = sbe_verdicts(rows)
vt = pd.DataFrame(
    [
        {
            "organism": o.split("/")[1].split("@")[0],
            "size": o.rsplit(":", 1)[1],
            "verdict": v.verdict,
            "why": "; ".join((v.seen.reasons if v.seen else ()) + v.unseen.reasons),
        }
        for o, v in verdicts.items()
    ]
)
vt.to_csv(OUT / "sbe_verdicts.csv", index=False)
vt.sort_values(["verdict", "organism", "size"])

## Per pool (T1 = `diag_v1` + `diag_v2`; which T2 pools are seen depends on the method)

In [ ]:
pools = df.groupby(["organism", "size", "pool"], observed=True).apply(rate, include_groups=False)
pools["rate"].unstack("pool").round(2)

## Per trait, unseen questions (the breadth check reads this)

The `seqkd` pool has no trait label, so it isn't in this table.

In [ ]:
traits = df[~df.seen].groupby(["organism", "size", "trait"], observed=True)
traits = traits.apply(rate, include_groups=False)["rate"].unstack("trait")
traits.columns = [TRAITS[int(t)] for t in traits.columns]
traits.to_csv(OUT / "sbe_traits_unseen.csv")
traits.round(2)

## Seed spread

Each seed is an independent pass over the 300 questions at temperature 1.0. The range across
seeds is the run-to-run noise in each organism's unseen rate.

In [ ]:
by_seed = df[~df.seen].groupby(["organism", "size", "seed"], observed=True)
by_seed = by_seed.apply(rate, include_groups=False)["rate"].unstack("seed")
by_seed["range"] = by_seed.max(axis=1) - by_seed.min(axis=1)
by_seed.round(3)

## Judge health: ambiguous labels and judge failures per organism

In [ ]:
health = (
    df.assign(ambiguous=df.outcome.eq("ambiguous"), judge_error=df.score.isna())
    .groupby(keys, observed=True)[["ambiguous", "judge_error"]]
    .sum()
)
health[(health.ambiguous > 0) | (health.judge_error > 0)]

# Figures

Each figure is saved to `analysis/out/sbe/<name>.pdf`, with a `.png` preview next to it.
Pick the ones the paper needs.

## Fig. 1: behaviour on unseen questions, every organism (single column)

Unseen means the questions this organism didn't train on, so the set differs by method (see
the summary). Fig. 5 is the like-for-like version: each organism minus base on the same
questions.

In [ ]:
def dot_rows(ax, table, col, lo_col, hi_col):
    ypos = {o: i for i, o in enumerate(reversed(ORDER))}
    offset = {"7B": 0.22, "14B": 0.0, "32B": -0.22}
    for (org, size), r in table.iterrows():
        y = ypos[org] + offset[size]
        ax.plot(
            [r[lo_col], r[hi_col]], [y, y], color=SIZE_COLOR[size], lw=0.9, solid_capstyle="round"
        )
        ax.plot(r[col], y, SIZE_MARKER[size], color=SIZE_COLOR[size], ms=4.2, mec=SURFACE, mew=0.6)
    ax.set_yticks(list(ypos.values()), list(ypos.keys()))
    ax.axhline(ypos["Prompt (+)"] - 0.5, color=MUTED, lw=0.6)  # controls above, implants below
    ax.set_xlim(-0.02, 1.02)
    ax.grid(axis="y", visible=False)


handles = [
    Line2D([], [], ls="", marker=SIZE_MARKER[s], color=SIZE_COLOR[s], ms=4.5, label=s)
    for s in SIZES
]


def size_legend(ax, **kw):
    kw = {"loc": "lower center", "bbox_to_anchor": (0.5, 1.0), "ncol": 3, **kw}
    ax.legend(handles=handles, frameon=False, handletextpad=0.2, columnspacing=1.2, **kw)


fig, ax = plt.subplots(figsize=(SINGLE_COL, 3.4))
dot_rows(ax, summary, "unseen_rate", "unseen_lo", "unseen_hi")
ax.set_xlabel("Share of answers judged impulsive (unseen questions)")
size_legend(ax)
save(fig, "fig1_sbe_unseen_by_organism")

## Fig. 2: seen vs unseen questions (single column)

This covers organisms that trained on part of the bank. Points below the diagonal express the
trait on their own training questions more than on new ones in the same style.

In [ ]:
st = summary.dropna(subset=["seen_rate"])
fig, ax = plt.subplots(figsize=(SINGLE_COL, 3.0))
ax.plot([0, 1], [0, 1], color=GRID, lw=1.0, zorder=0)
for (_org, size), r in st.iterrows():
    ax.plot(
        r.seen_rate,
        r.unseen_rate,
        SIZE_MARKER[size],
        color=SIZE_COLOR[size],
        ms=4.6,
        mec=SURFACE,
        mew=0.6,
    )
# One label per CLUSTER, placed from the cluster's own extent (data-driven, not hand-nudged):
# ICL's three k values and SFT/OCT/OCT (released) each sit on top of each other.
clusters = {
    "ICL (k=5, 15, 40)": ["ICL k=5", "ICL k=15", "ICL k=40"],
    "SeqKD": ["SeqKD"],
    "DPO": ["DPO"],
    "SFT, OCT,\nOCT (released)": ["SFT", "OCT", "OCT (released)"],
}
place = {  # (x, y, ha, va) relative to the cluster's bounding box
    "ICL (k=5, 15, 40)": ("xmin", "ymax", 0.0, 0.06, "left", "bottom"),
    "SeqKD": ("xmax", "ymax", 0.03, 0.02, "left", "bottom"),
    "DPO": ("xmin", "ymin", 0.0, -0.04, "left", "top"),
    "SFT, OCT,\nOCT (released)": ("xmin", "ymin", -0.04, -0.02, "right", "top"),
}
for text, members in clusters.items():
    sub = st[st.index.get_level_values("organism").isin(members)]
    box = {
        "xmin": sub.seen_rate.min(),
        "xmax": sub.seen_rate.max(),
        "ymin": sub.unseen_rate.min(),
        "ymax": sub.unseen_rate.max(),
    }
    xk, yk, dx, dy, ha, va = place[text]
    ax.text(box[xk] + dx, box[yk] + dy, text, fontsize=6.5, color=INK_2, ha=ha, va=va)
ax.set(
    xlim=(-0.02, 1.04),
    ylim=(-0.02, 1.04),
    aspect="equal",
    xlabel="Judged impulsive: seen (own training) questions",
    ylabel="Judged impulsive: unseen questions",
)
size_legend(ax)
save(fig, "fig2_sbe_seen_vs_unseen")

## Fig. 3: behaviour vs coherence (single column)

An organism can look impulsive because its text has fallen apart. The vertical line is the
coherence floor (70), which is a **placeholder** until calibration.

In [ ]:
fig, ax = plt.subplots(figsize=(SINGLE_COL, 2.7))
ax.axvline(SBE_COHERENCE_MIN, color=MUTED, lw=0.8)
floor_note = f"per-answer cut-off ({SBE_COHERENCE_MIN}):\nbelow it counts as absent"
ax.text(
    SBE_COHERENCE_MIN - 1.5, 0.45, floor_note, fontsize=6.5, color=MUTED, ha="right", va="center"
)
for (_org, size), r in summary.iterrows():
    ax.plot(
        r.coherence,
        r.unseen_rate,
        SIZE_MARKER[size],
        color=SIZE_COLOR[size],
        ms=4.6,
        mec=SURFACE,
        mew=0.6,
    )
dpo = summary.xs("DPO", level="organism")
ax.text(
    dpo.coherence.max() + 3, dpo.unseen_rate.mean(), "DPO", fontsize=6.5, color=INK_2, va="center"
)
ax.set(
    xlim=(0, 102),
    ylim=(-0.03, 1.05),
    xlabel="Mean judge coherence (0–100)",
    ylabel="Judged impulsive (unseen)",
)
size_legend(ax)
save(fig, "fig3_sbe_behaviour_vs_coherence")

## Fig. 4: per-trait breadth on unseen questions (full width)

Each cell is the share of answers judged impulsive on one trait's unseen questions, × 3 seeds.
That's 10 T1 questions per trait for SFT/DPO/OCT (30 answers). Organisms that didn't train on
the constitution and ICL-demo pools also have those 14 per trait (72 answers); ICL has the
constitution ones plus the examples not in its prompt. The sequential scale is a single hue.

In [ ]:
cmap = mpl.colors.LinearSegmentedColormap.from_list("seq", SEQUENTIAL)
rows_all = [o for o in ORDER if o in traits.index.get_level_values("organism")]
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL, 3.1), sharey=True)
meshes = []
for ax, size in zip(axes, SIZES, strict=True):
    # The SAME rows in every panel (sharey): an organism a size lacks (OCT (released) is
    # 7B-only) is a blank row marked "n/a", never a shifted row with the wrong label.
    t = traits.xs(size, level="size").reindex(rows_all)
    meshes.append(
        ax.pcolormesh(
            np.ma.masked_invalid(t.values),
            cmap=cmap,
            vmin=0,
            vmax=1,
            edgecolors=SURFACE,
            linewidth=1.0,
        )
    )
    for i, org in enumerate(rows_all):
        if t.loc[org].isna().all():
            ax.text(
                len(TRAITS) / 2,
                i + 0.5,
                "n/a",
                ha="center",
                va="center",
                fontsize=6.5,
                color=MUTED,
            )
    ax.set_xticks(np.arange(len(TRAITS)) + 0.5, TRAITS, rotation=60, ha="right", fontsize=6.5)
    ax.set_yticks(np.arange(len(rows_all)) + 0.5, rows_all)
    ax.set_ylim(len(rows_all), 0)
    ax.set_title(size)
    ax.grid(False)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
cb = fig.colorbar(meshes[-1], ax=axes, fraction=0.025, pad=0.01)
cb.set_label("Judged impulsive", color=INK_2)
cb.outline.set_visible(False)
save(fig, "fig4_sbe_trait_heatmap")

## Change compared with base

### Definitions

Let $p_\text{org}$ be the share of the organism's unseen answers judged impulsive, and
$p_\text{base}$ the same share for base **on the same questions** (base answered all 300).

- **Percentage-point (pp) gain over base** $= 100 \times (p_\text{org} - p_\text{base})$.
  - The **absolute** difference: how many more answers out of every 100 are judged impulsive
    than base's.
  - 0 means no change and +100 is the maximum (base never, organism always). Below 0 means
    less impulsive than base.
- **Relative % change over base** $= 100 \times (p_\text{org} - p_\text{base}) / p_\text{base}$.
  - The difference **as a multiple of base's own rate**: +100% means double base's rate,
    +1,000% means 11 times.
  - It has no upper limit, and it's undefined when $p_\text{base} = 0$.

### Which one to use

Base is almost never judged impulsive (the counts are printed below), so the two behave very
differently:

- **Percentage-point gain:**
  - It is defined everywhere, and its interval is Newcombe's (Wilson-based) interval for a
    difference of two proportions.
  - **Use this one in the paper.**
- **Relative % change:**
  - Where base scored 0 on an organism's questions, it's a division by zero.
  - Elsewhere it is dominated by how close base is to zero: a 2-answer change in base swings
    every organism by hundreds of percent.
  - It's shown for comparison with base smoothed as (k+1)/(n+2), and never as the headline.

In [ ]:
def newcombe(p1, l1, u1, p2, l2, u2):
    d = p1 - p2
    return d - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), d + np.sqrt(
        (u1 - p1) ** 2 + (p2 - l2) ** 2
    )


base_df = df[df.organism == "Base"]
gain = []
for (org, size), g in df[~df.seen].groupby(keys, observed=True):
    if org == "Base":
        continue
    o = rate(g)
    b = rate(base_df[(base_df["size"] == size) & base_df.item.isin(set(g.item))])
    lo, hi = newcombe(o.rate, o.lo, o.hi, b.rate, b.lo, b.hi)
    b_smooth = (b.present + 1) / (b.n + 2)
    gain.append(
        {
            "organism": org,
            "size": size,
            "base_present": int(b.present),
            "base_n": int(b.n),
            "gain_pp": 100 * (o.rate - b.rate),
            "gain_lo": 100 * lo,
            "gain_hi": 100 * hi,
            "relative_pct_raw": 100 * (o.rate - b.rate) / b.rate if b.rate > 0 else np.inf,
            "relative_pct_smoothed": 100 * (o.rate - b_smooth) / b_smooth,
        }
    )
gain = pd.DataFrame(gain).set_index(["organism", "size"])
gain["_o"] = [ORDER.index(o) for o in gain.index.get_level_values("organism")]
gain["_s"] = [SIZES.index(s) for s in gain.index.get_level_values("size")]
gain = gain.sort_values(["_o", "_s"]).drop(columns=["_o", "_s"])
gain.to_csv(OUT / "sbe_change_vs_base.csv")
gain.round(1)

## Fig. 5: percentage-point gain over base, unseen questions (single column; the paper version)

Gain $= 100 \times (p_\text{org} - p_\text{base})$, with base on the same questions. Bars are
95% Newcombe intervals.

In [ ]:
fig, ax = plt.subplots(figsize=(SINGLE_COL, 3.2))
g = gain.reset_index()
g = g[g.organism != "Base"]
rows_g = [o for o in ORDER if o != "Base"]
ypos = {o: i for i, o in enumerate(reversed(rows_g))}
offset = {"7B": 0.22, "14B": 0.0, "32B": -0.22}
ax.axvline(0, color=MUTED, lw=0.8)
for _, r in g.iterrows():
    y = ypos[r.organism] + offset[r["size"]]
    ax.plot(
        [r.gain_lo, r.gain_hi], [y, y], color=SIZE_COLOR[r["size"]], lw=0.9, solid_capstyle="round"
    )
    ax.plot(
        r.gain_pp,
        y,
        SIZE_MARKER[r["size"]],
        color=SIZE_COLOR[r["size"]],
        ms=4.2,
        mec=SURFACE,
        mew=0.6,
    )
ax.set_yticks(list(ypos.values()), list(ypos.keys()))
ax.axhline(ypos["Prompt (+)"] - 0.5, color=MUTED, lw=0.6)
ax.grid(axis="y", visible=False)
ax.set_xlim(-10, 102)
ax.set_xlabel("Gain over base (percentage points, unseen questions)")
size_legend(ax)
save(fig, "fig5_sbe_gain_over_base_pp")

## Fig. 5a: the same gain as a bar chart (single column)

The same numbers as Fig. 5, drawn as bars from zero with 95% Newcombe interval whiskers.

In [ ]:
fig, ax = plt.subplots(figsize=(SINGLE_COL, 3.4))
bar_h = 0.22
ax.axvline(0, color=MUTED, lw=0.8, zorder=3)
for _, r in g.iterrows():
    y = ypos[r.organism] + offset[r["size"]]
    ax.barh(
        y,
        r.gain_pp,
        height=bar_h,
        color=SIZE_COLOR[r["size"]],
        edgecolor=SURFACE,
        linewidth=0.6,
        zorder=2,
    )
    ax.plot([r.gain_lo, r.gain_hi], [y, y], color=INK_2, lw=0.7, zorder=4, solid_capstyle="butt")
ax.set_yticks(list(ypos.values()), list(ypos.keys()))
ax.axhline(ypos["Prompt (+)"] - 0.5, color=MUTED, lw=0.6)
ax.grid(axis="y", visible=False)
ax.set_xlim(-5, 102)
ax.set_xlabel("Gain over base (percentage points, unseen questions)")
bar_handles = [Patch(color=SIZE_COLOR[s], label=s) for s in SIZES]
ax.legend(
    handles=bar_handles,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncol=3,
    handlelength=1.0,
    handletextpad=0.4,
    columnspacing=1.2,
)
save(fig, "fig5a_sbe_gain_over_base_bars")

## Fig. 5b: relative % change over base, smoothed (single column; for comparison only)

Relative change $= 100 \times (p_\text{org} - \tilde p_\text{base}) / \tilde p_\text{base}$,
where $\tilde p_\text{base} = (k+1)/(n+2)$ is base's rate on the same questions, smoothed ($k$
of base's $n$ answers judged impulsive). The axis is logarithmic, and base is smoothed as (k+1)/(n+2), because base is only a handful
of answers (`base_present` in the table above, as few as 2 at 32B). Values
below zero (worse than base) can't be shown on a log axis, so they are listed underneath instead.

In [ ]:
fig, ax = plt.subplots(figsize=(SINGLE_COL, 3.2))
pos = g[g.relative_pct_smoothed > 0]
for _, r in pos.iterrows():
    y = ypos[r.organism] + offset[r["size"]]
    ax.plot(
        r.relative_pct_smoothed,
        y,
        SIZE_MARKER[r["size"]],
        color=SIZE_COLOR[r["size"]],
        ms=4.2,
        mec=SURFACE,
        mew=0.6,
    )
ax.set_xscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}%"))
ax.set_yticks(list(ypos.values()), list(ypos.keys()))
ax.axhline(ypos["Prompt (+)"] - 0.5, color=MUTED, lw=0.6)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Relative change over base (%, log scale; base smoothed)")
size_legend(ax)
save(fig, "fig5b_sbe_relative_change_vs_base")
print("Not shown (at or below base after smoothing):")
g[g.relative_pct_smoothed <= 0][["organism", "size", "relative_pct_smoothed"]].round(1)

## Fig. 6: change in coherence compared with base (single column)

Coherence is the judge's 0–100 score for whether an answer makes sense, averaged over all 300
questions (seen and unseen together).

- **Coherence change** $= \bar c_\text{org} - \bar c_\text{base}$, in coherence points on that
  0–100 scale.
  - Both means are over the **same 300 questions**.
  - There's no ×100 here: coherence is already on a 0–100 scale, unlike the 0–1 rates in
    Fig. 5.
  - 0 means as coherent as base, and negative means less coherent.
- **Interval:** 95%, paired by question. Each question's mean over its 3 seeds is taken for the
  organism and for base, the per-question differences are averaged, and the normal
  approximation over the ~300 questions gives the interval.

The verdict's coherence floor (70) is an absolute level, not a change, so it isn't drawn here;
Fig. 3 shows it.

In [ ]:
coh_item = df.groupby(["organism", "size", "item"], observed=True)["coherence"].mean()
base_item = coh_item.xs("Base", level="organism")
coh_change = []
for (org, size), s in coh_item.groupby(level=["organism", "size"], observed=True):
    if org == "Base":
        continue
    diff = (s.droplevel(["organism", "size"]) - base_item.xs(size, level="size")).dropna()
    half = 1.959964 * diff.std(ddof=1) / np.sqrt(len(diff))
    coh_change.append(
        {
            "organism": org,
            "size": size,
            "questions": len(diff),
            "change": diff.mean(),
            "lo": diff.mean() - half,
            "hi": diff.mean() + half,
        }
    )
coh_change = pd.DataFrame(coh_change)
coh_change["_o"] = coh_change.organism.map(ORDER.index)
coh_change["_s"] = coh_change["size"].map(SIZES.index)
coh_change = coh_change.sort_values(["_o", "_s"]).drop(columns=["_o", "_s"])
coh_change.to_csv(OUT / "sbe_coherence_change_vs_base.csv", index=False)

fig, ax = plt.subplots(figsize=(SINGLE_COL, 3.4))
ax.axvline(0, color=MUTED, lw=0.8, zorder=3)
for _, r in coh_change.iterrows():
    y = ypos[r.organism] + offset[r["size"]]
    ax.barh(
        y,
        r.change,
        height=bar_h,
        color=SIZE_COLOR[r["size"]],
        edgecolor=SURFACE,
        linewidth=0.6,
        zorder=2,
    )
    ax.plot([r.lo, r.hi], [y, y], color=INK_2, lw=0.7, zorder=4, solid_capstyle="butt")
ax.set_yticks(list(ypos.values()), list(ypos.keys()))
ax.axhline(ypos["Prompt (+)"] - 0.5, color=MUTED, lw=0.6)
ax.grid(axis="y", visible=False)
ax.set_xlim(-85, 5)
ax.set_xlabel("Change in coherence vs base (points, 0–100 scale)")
ax.legend(
    handles=bar_handles,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncol=3,
    handlelength=1.0,
    handletextpad=0.4,
    columnspacing=1.2,
)
save(fig, "fig6_sbe_coherence_change_vs_base")
coh_change.round(1)

## Files written

The CSV tables are in `analysis/out/sbe/`, next to the figures, if the paper needs them.

In [ ]:
sorted(p.name for p in OUT.iterdir())